# Bozyazı deniz seviyesi analizi — Colab

Bu defter tüm zinciri baştan sona çalıştırır: veriyi TUDES portalından
indirir, ayıklar, harmonik analizi yapar ve figürleri üretir.

**Önce çalışma zamanını yüksek RAM'e alın.** `05_harmonik_analiz.py`
15 dakikalık çözünürlükte 16 yıllık kayıtla çalışırken UTide'ın en küçük
kareler tasarım matrisi ~9 GB istiyor. Standart Colab (12 GB) sınırda
kalır; yüksek-RAM çalışma zamanı rahat çalıştırır.

`Çalışma zamanı → Çalışma zamanı türünü değiştir → Yüksek RAM`

## 1. Kurulum

In [ ]:
!git clone https://github.com/adzetto/marine_analysis.git
%cd marine_analysis/su-seviyesi
!pip install -q -r requirements.txt

LaTeX dizgisi isteğe bağlıdır. Kurulmazsa figürler matplotlib'in kendi
matematik dizgisiyle üretilir (betikler bunu kendisi algılar). Yayın
kalitesi isteniyorsa aşağıdaki hücreyi çalıştırın — birkaç dakika sürer.

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super > /dev/null
print('latex kuruldu')

## 2. Kaynakları göster

Harmonik çözümün sığıp sığmayacağını baştan görmek için.

In [ ]:
import psutil, os
gb = psutil.virtual_memory().total / 1e9
print(f'toplam RAM : {gb:.1f} GB')
print(f'CPU        : {os.cpu_count()} cekirdek')
if gb < 20:
    print('\nUYARI: 05_harmonik_analiz.py tum kayitta ~9 GB istiyor.')
    print('Yuksek RAM calisma zamanina gecmeniz onerilir.')

## 3. Veriyi indir

2010'dan bugüne, 55 günlük parçalar hâlinde (~110 istek, birkaç dakika).
Parçalar `data/ham/` altında önbelleğe alınır; hücre tekrar çalıştırılırsa
yalnız eksikler indirilir.

In [ ]:
!python -u 01_tudes_indir.py

## 4. Birleştir ve veriyi tanı

In [ ]:
!python -u 02_veri_birlestir.py

## 5. Ayıkla

Sıçrama, sürekli blok ve takılmış sensör ölçümlerini atar, 1 günden kısa
boşlukları doldurur, MSL'i hesaplar. Ardından ayıklamanın doğru şeyi
sildiği sınanır.

In [ ]:
!python -u 03_veri_ayikla.py
!python -u 04_ayiklama_dogrula.py

## 6. Harmonik analiz

Ağır adım bu. Dört dönem için UTide çözümü ve makaleyle karşılaştırma.

In [ ]:
!python -u 05_harmonik_analiz.py

## 7. Gelgit düzeyleri ve gelgit dışı bileşen

In [ ]:
!python -u 06_gelgit_seviyeleri.py
!python -u 07_non_tidal.py

## 8. Sonuçları göster

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image

for f in sorted(glob.glob('tables/*.csv')):
    print('=' * 70); print(f); print('=' * 70)
    display(pd.read_csv(f))

for f in sorted(glob.glob('figures/*.png')):
    print(f)
    display(Image(f))

## 9. Çıktıları indir

In [ ]:
!zip -qr bozyazi_sonuclar.zip tables figures data/*.dat
from google.colab import files
files.download('bozyazi_sonuclar.zip')